In [1]:
# run the hydrologic model (HBV) based on various parameter sets, including
# (1) run model and generate model output files
# (2) generate evaluation results
# (3) train an emulator based on all basins
# (4) generate new parameter sets based on the emulator

In [2]:
import glob, os, sys, time, toml, multiprocessing
from datetime import datetime
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

sys.path.append("../auxiliary/")
from data_processing import *
from parameter import *
from run_model import *
from evaluate import cal_metrics

sys.path.append("../HBV/")
from HBV import load_HBV_output, run_HBV_OneBasin_OneParamSet

# Target iteration
Simulate this iteration and generate parameters for the next iteration

In [3]:
iter = 0
ncpu = 36  # if ncpu>1, parallelization will be used
run_HBV = False # if False, HBV won't be run. this can be used when you know model simulations have been generated

# Load data

In [4]:
# load configuration
config_file = "config_LSEexp1.toml"
config = toml.load(config_file)

inpath_data = config['data']['inpath_data']
file_parameter_list = config['data']['file_parameter_list']

outpath_parent = config["output"]["outpath_parent"]

train_run_period = config["dates"]["train_run_period"]
train_eval_period = config["dates"]["train_eval_period"]
eval_indexStart, eval_indexEnd = get_index_of_nested_period(train_run_period, train_eval_period)

sampling_method = config["emulator"]["sampling_method"]
num_init = config["emulator"]["num_init"]
objfunc = config["emulator"]["objfunc"]
num_per_iter = config["emulator"]["num_per_iter"]
if iter == 0:
    num_trial = config["emulator"]["num_init"]
else:
    num_trial = num_per_iter


display(config)

{'intro': {'author': 'Guoqiang Tang',
  'name': 'HBV LSE calibration',
  'date': '2024-12'},
 'HPC': {'projectCode': 'P08010000'},
 'data': {'file_parameter_list': '/glade/u/home/guoqiang/calibration/HBVcalib/data/HBV_v1.1_param_bounds.csv',
  'inpath_data': '/glade/campaign/cgd/tss/people/guoqiang/HBV_CAMELS/data_Yalan'},
 'output': {'outpath_parent': '/glade/campaign/cgd/tss/people/guoqiang/HBV_CAMELS/LSEexp1'},
 'dates': {'alldata_period': ['1980-10-01', '2014-09-30'],
  'train_run_period': ['1999-10-01', '2008-09-30'],
  'train_eval_period': ['2000-10-01', '2008-09-30'],
  'test_run_period': ['1986-10-01', '1999-09-30'],
  'test_eval_period': ['1989-10-01', '2008-09-30']},
 'emulator': {'objfunc': 'normNSE',
  'sampling_method': 'lh',
  'num_init': 400,
  'num_per_iter': 100,
  'num_iter': 10}}

In [5]:
# forcing, flow, and basin data
startDate = train_run_period[0]
endDate = train_run_period[1]
print("run model period:", startDate, "-", endDate)

forcing_train, target_train, attr_train, date_train, forcingLst, allbasin_IDs, attrLst = load_Yalan_data(inpath_data, startDate, endDate)
print("forcing shape", forcing_train.shape)

num_basin = len(allbasin_IDs)
print("basin number", num_basin)

run model period: 1999-10-01 - 2008-09-30
forcing shape (531, 3288, 3)
basin number 531


In [6]:
# parameter list containing upper / lower bounds
df_param_bound = pd.read_csv(file_parameter_list)
df_param_bound.head()

,Parameter,Default,Lower,Upper,Method
0,parBETA,3.5000,1.000,6.0,Multiplicative
1,parFC,525.0000,50.000,1000.0,Multiplicative
2,parK0,0.4750,0.050,0.9,Multiplicative
3,parK1,0.2550,0.010,0.5,Multiplicative
4,parK2,0.1005,0.001,0.2,Multiplicative


# Run the model 

## Serial version

This can be useful diagnosis, but is not used in production  
It contains functions from run_modelflow_OneBasin_OneParamSet

In [7]:
# for tarbasin in range(num_basin):

#     gageID = allbasin_IDs[tarbasin]
#     outpath_basin = f'{outpath_parent}/{gageID}/outputs/'
#     os.makedirs(outpath_basin, exist_ok=True)

#     Ptar = forcing_train[tarbasin, :, forcingLst.index('prcp')][:, np.newaxis]
#     Ttar = forcing_train[tarbasin, :, forcingLst.index('tmean')][:, np.newaxis]
#     PETtar = forcing_train[tarbasin, :, forcingLst.index('pet')][:, np.newaxis]

#     for tarparam in range(num_init):

#         # target parameter set
#         file_paramset = f'{outpath_parent}/{gageID}/param_sets/paramset_iter{iter}_trial{tarparam}.pkl'

#         # run HBV
#         outfile_hbv = f'{outpath_basin}/hbv_iter{iter}_trial{tarparam}.pkl'
#         run_HBV_OneBasin_OneParamSet(Ptar, Ttar, PETtar, file_paramset, outfile_hbv)

#         # evaluate model output
#         outfile_metric = f'{outpath_basin}/eval_iter{iter}_trial{tarparam}.csv'
#         hbv_out = load_HBV_output(outfile_hbv)
#         eval_sim = hbv_out['flow_sim'][eval_indexStart:eval_indexEnd+1]
#         eval_obs = target_train[tarbasin, :, 0][eval_indexStart:eval_indexEnd+1]
#         df_tar_metric = cal_metrics(eval_obs, eval_sim, outfile_metric)

#         # print output
#         kge=df_tar_metric['kge2009'].values[0]
#         nse=df_tar_metric['nse'].values[0]
#         print(f"// Basin-{tarbasin}: {gageID}; ParamSet-{tarparam} // -> KGE={kge:.2f}; NSE={nse:0.2f}")

## Parallel version
run_modelflow_OneBasin_OneParamSet is a high-level function wrapping up many components

In [8]:
if run_HBV == False:
    print('Do not run HBV because run_HBV is False')

else:
    
    # log file to save printouts
    outpath_log = f"{outpath_parent}/logs"
    os.makedirs(outpath_log, exist_ok=True)
    current_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    log_file_path = f"{outpath_log}/workflow_{current_time}.log"
    print("See printout information in", log_file_path)
    
    with open(log_file_path, "w") as log_file:
        # Redirect stdout to the log file
        original_stdout = sys.stdout
        sys.stdout = log_file
    
        try:
            # Running the model
    
            t01 = time.time()
            for tarbasin in range(num_basin):
                t11 = time.time()
    
                gageID = allbasin_IDs[tarbasin]
                outpath_basin = f"{outpath_parent}/{gageID}/outputs/"
                os.makedirs(outpath_basin, exist_ok=True)
    
                Ptar = forcing_train[tarbasin, :, forcingLst.index("prcp")][:, np.newaxis]
                Ttar = forcing_train[tarbasin, :, forcingLst.index("tmean")][:, np.newaxis]
                PETtar = forcing_train[tarbasin, :, forcingLst.index("pet")][:, np.newaxis]
    
                eval_obs = target_train[tarbasin, :, 0][eval_indexStart : eval_indexEnd + 1]
    
                if ncpu == 1:
                    # serial
                    for tarparam in range(num_trial):
                        run_modelflow_OneBasin_OneParamSet(tarparam, iter, gageID, tarbasin, Ptar, Ttar, PETtar,
                                                           outpath_parent, outpath_basin, eval_indexStart, eval_indexEnd, eval_obs)
                else:
                    # parallel
                    alltrial = [(tarparam, iter, gageID, tarbasin, Ptar, Ttar, PETtar, outpath_parent, outpath_basin, 
                                 eval_indexStart, eval_indexEnd, eval_obs) for tarparam in range(num_trial)]
                    
                    with multiprocessing.Pool(processes=ncpu) as pool:
                        pool.starmap(run_modelflow_OneBasin_OneParamSet, alltrial)
    
                t12 = time.time()
                print(f"Time cost for basin-{tarbasin} gageID (s):", t12 - t11)
    
            t02 = time.time()
            print(f"Time cost for all basins (s):", t02 - t01)
        finally:
            # Restore stdout after the block is complete
            sys.stdout = original_stdout

Do not run HBV because run_HBV is False


## File post-processing
merge parameter / evaluation files for one iteration to one file to faciliate later analysis

In [9]:

if ncpu == 1:
    # Serial execution with progress bar
    for tarbasin in tqdm(range(num_basin), desc="Merging param/eval for all basins"):
        gageID = allbasin_IDs[tarbasin]
        path_archive = f"{outpath_parent}/{gageID}/"
        merge_parameter_metric_OneBasin_OneIter(path_archive, iter, num_trial)

else:
    args = [(f"{outpath_parent}/{allbasin_IDs[tarbasin]}/", iter, num_trial) for tarbasin in range(num_basin) ]
    # Use multiprocessing with a progress bar
    with multiprocessing.Pool(processes=ncpu) as pool:
        list(tqdm(
            pool.starmap(merge_parameter_metric_OneBasin_OneIter, args),
            total=len(args),
            desc="Merging param/eval for all basins"
        ))


Merging param/eval for all basins:   0%|          | 0/531 [00:00<?, ?it/s]

# Large-sample emulator: Train and Optimization

## data preparation

In [10]:
outpath_lse = f'{outpath_parent}/LargeSampleEmulator'
os.makedirs(outpath_lse, exist_ok=True)
print('LSE outpath:', outpath_lse)

LSE outpath: /glade/campaign/cgd/tss/people/guoqiang/HBV_CAMELS/LSEexp1/LargeSampleEmulator


In [11]:
# generate files containing parameter, metric, and basinid information
iterend = iter+1
df_param, df_metric, df_basinid = load_allbasin_params_metrics_AllIter(outpath_parent, outpath_lse, allbasin_IDs, iterend)

Number of nan samples: 0
Number of original parameter sets: 212400
Number of final parameter sets: 212400


In [12]:
# prepare attribute

# attribute list used in CTSM/SUMMA original studies
attrLst_Xie2021 = ['elev_mean', 'slope_mean', 'area_gages2', 'p_mean', 'pet_mean', 'aridity', 'p_seasonality', 'frac_snow', 
                   'high_prec_freq', 'high_prec_dur', 'low_prec_freq', 'low_prec_dur', 'forest_frac', 'lai_max', 'lai_diff', 
                   'dom_land_cover', 'dom_land_cover_frac', 'soil_depth_pelletier', 'soil_depth_statgso', 'soil_porosity', 
                   'soil_conductivity', 'max_water_content', 'sand_frac', 'silt_frac', 'clay_frac', 'carb_rocks_frac', 'geol_permeability']

# remove attributes not included in Xie2021
flag = np.ones(len(attrLst))
for i in range(len(attrLst)):
    if not attrLst[i] in attrLst_Xie2021:
        flag[i] = 0

attrLst_filtered = list(np.array(attrLst)[flag==1])
print('Attribute - Xie2021 number:', len(attrLst_Xie2021))
print('Attribute - YSong number:', len(attrLst))
print('Attribute - YSong_filtered number:', len(attrLst_filtered))
for a in attrLst_Xie2021:
    if not a in attrLst_filtered:
        print(f'Missing {a} in YSong_filtered compared to Xie2021')

# make a dataframe
df_att = pd.DataFrame(attr_train[:,flag==1], columns=attrLst_filtered)
df_att['hru_id'] = allbasin_IDs

Attribute - Xie2021 number: 27
Attribute - YSong number: 35
Attribute - YSong_filtered number: 24
Missing forest_frac in YSong_filtered compared to Xie2021
Missing soil_depth_statgso in YSong_filtered compared to Xie2021
Missing carb_rocks_frac in YSong_filtered compared to Xie2021


In [13]:
# Prepare model input
df_input = df_param.copy()
df_input["hru_id"] = df_basinid["hru_id"]
df_input = df_input.merge(df_att[attrLst_filtered + ["hru_id"]], on="hru_id", how="left")
df_input = df_input.drop(["hru_id"], axis=1)

inputnames = list(df_param.columns) + attrLst_filtered
print(f'Input features number: {len(inputnames)} \n', inputnames)

x_all = df_input[inputnames].values.copy()
print("Input shape:", x_all.shape)

Input features number: 40 
 ['parBETA', 'parFC', 'parK0', 'parK1', 'parK2', 'parLP', 'parPERC', 'parUZL', 'parTT', 'parCFMAX', 'parCFR', 'parCWH', 'parBETAET', 'parC', 'rout_a', 'rout_b', 'p_mean', 'pet_mean', 'p_seasonality', 'frac_snow', 'aridity', 'high_prec_freq', 'high_prec_dur', 'low_prec_freq', 'low_prec_dur', 'elev_mean', 'slope_mean', 'area_gages2', 'lai_max', 'lai_diff', 'dom_land_cover_frac', 'dom_land_cover', 'soil_depth_pelletier', 'soil_porosity', 'soil_conductivity', 'max_water_content', 'sand_frac', 'silt_frac', 'clay_frac', 'geol_permeability']
Input shape: (212400, 40)


In [14]:
# Prepare model training target
y_all = get_objective_function(df_metric, objfunc)
print('Output shape:', y_all.shape)

Output shape: (212400,)


## Train RF emulator

In [20]:
%%time
# Train a random forest emulator
iterend = iter + 1
outfile_lse = f'{outpath_lse}/RF_emulator_for_iter{iterend}.pkl'
if os.path.isfile(outfile_lse):
    with open(outfile_lse, 'rb') as file:
        em_model = pickle.load(file)
else:
    modelconfig = {'n_estimators': 100, 'random_state': 42, 'max_depth': 40}
    em_model = RandomForestRegressor(**modelconfig, n_jobs=ncpu)
    em_model.fit(x_all, y_all)
    with open(outfile_lse, 'wb') as file:
        pickle.dump(em_model, file)

# print em_model training accuracy
y_all_pred = em_model.predict(x_all)
print('rmse of em:', np.mean( (y_all_pred - y_all) ** 2 ) ** 0.5)


rmse of em: 0.028623062655792647
CPU times: user 7.84 s, sys: 1.24 s, total: 9.08 s
Wall time: 2.35 s


## Generate optimized parameter values for each basin

In [21]:
# check parameter match
if np.any( df_param.columns != df_param_bound['Parameter'] ):
    sys.exit('df_param and df_param_bound parameters do not match')

In [ ]:
# optimize for each basin

iterend = iter + 1
modelparam_lb = df_param_bound['Lower'].values
modelparam_ub = df_param_bound['Upper'].values
param_calibflag = np.zeros( len(df_input.columns) )
param_calibflag[:len(modelparam_lb)]=1
param_names = df_input.columns.values

if ncpu > 1:
    args = [
        (
            f'{outpath_parent}/{allbasin_IDs[i]}/param_sets',  # outpath_param
            iterend, 
            em_model, 
            np.hstack([modelparam_lb, df_att[attrLst_filtered].iloc[i].values]),  # param_lb
            np.hstack([modelparam_ub, df_att[attrLst_filtered].iloc[i].values]),  # param_ub
            param_calibflag, 
            param_names, 
            num_per_iter, 
        )
        for i in range(num_basin)
    ]

    with multiprocessing.Pool(processes=ncpu) as pool:
        pool.starmap(generate_optimized_paramsets, args)
        
else:

    for i in range(len(allbasin_IDs)):
        outpath_param = f'{outpath_parent}/{allbasin_IDs[i]}/param_sets'
        os.makedirs(outpath_param, exist_ok=True)
        
        basin_attr = df_att[attrLst_filtered].iloc[i].values
        param_lb = np.hstack([modelparam_lb, basin_attr])
        param_ub = np.hstack([modelparam_ub, basin_attr])
        
        generate_optimized_paramsets(outpath_param, iterend, em_model, param_lb, param_ub, param_calibflag, param_names, num_per_iter, num_objfunc=1, times=3)

Run GA 300 times and extract the best 100 solutions
Run GA 300 times and extract the best 100 solutions


/glade/work/guoqiang/conda-envs/npl-2024a-tgq/lib/python3.11/site-packages/sklearn/ensemble/_base.py:214: UserWarning: Loky-backed parallel loops cannot be called in a multiprocessing, setting n_jobs=1
  n_jobs = min(effective_n_jobs(n_jobs), n_estimators)
/glade/work/guoqiang/conda-envs/npl-2024a-tgq/lib/python3.11/site-packages/sklearn/ensemble/_base.py:214: UserWarning: Loky-backed parallel loops cannot be called in a multiprocessing, setting n_jobs=1
  n_jobs = min(effective_n_jobs(n_jobs), n_estimators)
/glade/work/guoqiang/conda-envs/npl-2024a-tgq/lib/python3.11/site-packages/sklearn/ensemble/_base.py:214: UserWarning: Loky-backed parallel loops cannot be called in a multiprocessing, setting n_jobs=1
  n_jobs = min(effective_n_jobs(n_jobs), n_estimators)
/glade/work/guoqiang/conda-envs/npl-2024a-tgq/lib/python3.11/site-packages/sklearn/ensemble/_base.py:214: UserWarning: Loky-backed parallel loops cannot be called in a multiprocessing, setting n_jobs=1
  n_jobs = min(effective_n_

Run GA 300 times and extract the best 100 solutions


/glade/work/guoqiang/conda-envs/npl-2024a-tgq/lib/python3.11/site-packages/sklearn/ensemble/_base.py:214: UserWarning: Loky-backed parallel loops cannot be called in a multiprocessing, setting n_jobs=1
  n_jobs = min(effective_n_jobs(n_jobs), n_estimators)
/glade/work/guoqiang/conda-envs/npl-2024a-tgq/lib/python3.11/site-packages/sklearn/ensemble/_base.py:214: UserWarning: Loky-backed parallel loops cannot be called in a multiprocessing, setting n_jobs=1
  n_jobs = min(effective_n_jobs(n_jobs), n_estimators)
/glade/work/guoqiang/conda-envs/npl-2024a-tgq/lib/python3.11/site-packages/sklearn/ensemble/_base.py:214: UserWarning: Loky-backed parallel loops cannot be called in a multiprocessing, setting n_jobs=1
  n_jobs = min(effective_n_jobs(n_jobs), n_estimators)
/glade/work/guoqiang/conda-envs/npl-2024a-tgq/lib/python3.11/site-packages/sklearn/ensemble/_base.py:214: UserWarning: Loky-backed parallel loops cannot be called in a multiprocessing, setting n_jobs=1
  n_jobs = min(effective_n_

Run GA 300 times and extract the best 100 solutions


/glade/work/guoqiang/conda-envs/npl-2024a-tgq/lib/python3.11/site-packages/sklearn/ensemble/_base.py:214: UserWarning: Loky-backed parallel loops cannot be called in a multiprocessing, setting n_jobs=1
  n_jobs = min(effective_n_jobs(n_jobs), n_estimators)
/glade/work/guoqiang/conda-envs/npl-2024a-tgq/lib/python3.11/site-packages/sklearn/ensemble/_base.py:214: UserWarning: Loky-backed parallel loops cannot be called in a multiprocessing, setting n_jobs=1
  n_jobs = min(effective_n_jobs(n_jobs), n_estimators)
/glade/work/guoqiang/conda-envs/npl-2024a-tgq/lib/python3.11/site-packages/sklearn/ensemble/_base.py:214: UserWarning: Loky-backed parallel loops cannot be called in a multiprocessing, setting n_jobs=1
  n_jobs = min(effective_n_jobs(n_jobs), n_estimators)
/glade/work/guoqiang/conda-envs/npl-2024a-tgq/lib/python3.11/site-packages/sklearn/ensemble/_base.py:214: UserWarning: Loky-backed parallel loops cannot be called in a multiprocessing, setting n_jobs=1
  n_jobs = min(effective_n_

... (9 outputs left out)

Run GA 300 times and extract the best 100 solutions


/glade/work/guoqiang/conda-envs/npl-2024a-tgq/lib/python3.11/site-packages/sklearn/ensemble/_base.py:214: UserWarning: Loky-backed parallel loops cannot be called in a multiprocessing, setting n_jobs=1
  n_jobs = min(effective_n_jobs(n_jobs), n_estimators)
/glade/work/guoqiang/conda-envs/npl-2024a-tgq/lib/python3.11/site-packages/sklearn/ensemble/_base.py:214: UserWarning: Loky-backed parallel loops cannot be called in a multiprocessing, setting n_jobs=1
  n_jobs = min(effective_n_jobs(n_jobs), n_estimators)
/glade/work/guoqiang/conda-envs/npl-2024a-tgq/lib/python3.11/site-packages/sklearn/ensemble/_base.py:214: UserWarning: Loky-backed parallel loops cannot be called in a multiprocessing, setting n_jobs=1
  n_jobs = min(effective_n_jobs(n_jobs), n_estimators)
/glade/work/guoqiang/conda-envs/npl-2024a-tgq/lib/python3.11/site-packages/sklearn/ensemble/_base.py:214: UserWarning: Loky-backed parallel loops cannot be called in a multiprocessing, setting n_jobs=1
  n_jobs = min(effective_n_

Run GA 300 times and extract the best 100 solutions


/glade/work/guoqiang/conda-envs/npl-2024a-tgq/lib/python3.11/site-packages/sklearn/ensemble/_base.py:214: UserWarning: Loky-backed parallel loops cannot be called in a multiprocessing, setting n_jobs=1
  n_jobs = min(effective_n_jobs(n_jobs), n_estimators)
/glade/work/guoqiang/conda-envs/npl-2024a-tgq/lib/python3.11/site-packages/sklearn/ensemble/_base.py:214: UserWarning: Loky-backed parallel loops cannot be called in a multiprocessing, setting n_jobs=1
  n_jobs = min(effective_n_jobs(n_jobs), n_estimators)
/glade/work/guoqiang/conda-envs/npl-2024a-tgq/lib/python3.11/site-packages/sklearn/ensemble/_base.py:214: UserWarning: Loky-backed parallel loops cannot be called in a multiprocessing, setting n_jobs=1
  n_jobs = min(effective_n_jobs(n_jobs), n_estimators)
/glade/work/guoqiang/conda-envs/npl-2024a-tgq/lib/python3.11/site-packages/sklearn/ensemble/_base.py:214: UserWarning: Loky-backed parallel loops cannot be called in a multiprocessing, setting n_jobs=1
  n_jobs = min(effective_n_